# Data Exploration

This notebook loads satellite telemetry data, computes basic statistics, and visualizes key features.

## Prerequisites

```bash
pip install -e ".[notebooks]"
```

Place your data under `data/processed/merged_data/simulations_year/` (see `docs/data_format.md`).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sat_anomaly.config import get_lstm_config
from sat_anomaly.data.loader import load_fault_free_data, get_data_statistics
from sat_anomaly.data.preprocessor import normalize_features

print("Imports OK")

## Load fault-free data

In [ ]:
config = get_lstm_config()
data = load_fault_free_data(config['data_path'])
stats = get_data_statistics(data)
stats

## Feature distributions

In [ ]:
import numpy as np

numeric_cols = data.select_dtypes(include=[np.number]).columns
feature_cols = [c for c in numeric_cols if c not in ['time_ns', 'time_s', 'label_any_fault']]

fig, axes = plt.subplots(6, 4, figsize=(16, 14))
axes = axes.ravel()
for i, col in enumerate(feature_cols):
    if i >= len(axes):
        break
    axes[i].hist(data[col].dropna(), bins=50, alpha=0.7)
    axes[i].set_title(col, fontsize=9)
for j in range(len(feature_cols), len(axes)):
    axes[j].axis('off')
plt.suptitle('Feature Distributions (Fault-Free Data)')
plt.tight_layout()
plt.show()

## Time-series visualization

In [ ]:
# Plot a single simulation run
seq_ids = data['sequence_id'].unique()
sample = data[data['sequence_id'] == seq_ids[0]].head(5000)

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
axes[0].plot(sample['time_s'], sample['rw0_omega'], label='rw0_omega')
axes[0].plot(sample['time_s'], sample['rw1_omega'], label='rw1_omega')
axes[0].plot(sample['time_s'], sample['rw2_omega'], label='rw2_omega')
axes[0].set_ylabel('Angular Velocity (rad/s)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(sample['time_s'], sample['battery_voltage'], label='voltage', color='C3')
axes[1].set_ylabel('Battery Voltage (V)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(sample['time_s'], sample['bus_power'], label='bus_power', color='C4')
axes[2].set_ylabel('Bus Power (W)')
axes[2].set_xlabel('Time (s)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('Fault-Free Telemetry Sample')
plt.tight_layout()
plt.show()

## Normalization check

In [ ]:
normed, scaler = normalize_features(data)
print(f'Normalized range: [{normed[feature_cols].min().min():.4f}, {normed[feature_cols].max().max():.4f}]')
normed[feature_cols].describe()